# SHL Grammar Scoring: Approach A (End-to-End Pipeline)

This notebook contains the complete Approach A solution. It is designed to be fully executable on Google Colab or Kaggle.

## 1. Environment Setup

In [ ]:
# Install required libraries
!pip install -q pandas numpy scikit-learn librosa soundfile openai-whisper spacy sentence-transformers catboost xgboost tqdm
!python -m spacy download en_core_web_sm
!apt-get update && apt-get install -y ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 8.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 107.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import torch
import whisper
import spacy
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.metrics import root_mean_squared_error
from scipy.stats import pearsonr
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

# CONFIGURATION
# Assuming Dataset_Final is uploaded directly to the root of your Google Drive
DATASET_DIR = "/content/drive/MyDrive/Dataset_Final"

# Fallback for local Windows execution
if not os.path.exists(DATASET_DIR):
    DATASET_DIR = r"C:\Users\Kumar Abhishek\OneDrive\Desktop\SHL\Dataset_Final"

TRAIN_CSV = os.path.join(DATASET_DIR, "train.csv")
TEST_CSV = os.path.join(DATASET_DIR, "test.csv")
TRAIN_DIR = os.path.join(DATASET_DIR, "train")
TEST_DIR = os.path.join(DATASET_DIR, "test")

# Ensure output directories exist
os.makedirs("outputs", exist_ok=True)
os.makedirs("cache", exist_ok=True)

## 2. Dataset Loading & Validation

In [ ]:
df_train = pd.read_csv(TRAIN_CSV)
df_test = pd.read_csv(TEST_CSV)

df_train['audio_path'] = df_train['filename'].apply(lambda x: os.path.join(TRAIN_DIR, x))
df_test['audio_path'] = df_test['filename'].apply(lambda x: os.path.join(TEST_DIR, x))

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)
display(df_train.head(3))

Train shape: (769, 3)
Test shape: (216, 3)


,filename,label,audio_path
0,audio_192.wav,3.0,/content/drive/MyDrive/Dataset_Final/train/aud...
1,audio_436.wav,3.0,/content/drive/MyDrive/Dataset_Final/train/aud...
2,audio_447.wav,3.5,/content/drive/MyDrive/Dataset_Final/train/aud...


## 3. Audio Transcription (Whisper)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
asr_model = whisper.load_model("tiny").to(device)

def transcribe_dataset(df, cache_path):
    if os.path.exists(cache_path):
        print("Loading cached transcripts...")
        return pd.read_csv(cache_path)

    transcripts = []
    for path in tqdm(df['audio_path'], desc="Transcribing"):
        try:
            res = asr_model.transcribe(path, fp16=(device=="cuda"))
            transcripts.append(res["text"].strip())
        except Exception as e:
            transcripts.append("")

    df['transcript'] = transcripts
    df.to_csv(cache_path, index=False)
    return df

df_train = transcribe_dataset(df_train, "cache/train_transcripts.csv")
df_test = transcribe_dataset(df_test, "cache/test_transcripts.csv")

Using device: cuda


100%|██████████████████████████████████████| 72.1M/72.1M [00:00<00:00, 262MiB/s]


Transcribing:   0%|          | 0/769 [00:00<?, ?it/s]

Transcribing:   0%|          | 0/216 [00:00<?, ?it/s]

## 4. Feature Extraction (Linguistic & Embeddings)

In [ ]:
nlp = spacy.load("en_core_web_sm")
emb_model = SentenceTransformer("all-MiniLM-L6-v2")

def extract_features(df):
    word_counts, sent_counts, mean_w_lens, unique_words = [], [], [], []

    for text in tqdm(df['transcript'].fillna(""), desc="Linguistic"):
        doc = nlp(text)
        words = [token.text for token in doc if not token.is_punct]
        sents = list(doc.sents)

        wc = len(words)
        sc = len(sents) if len(sents) > 0 else 1

        word_counts.append(wc)
        sent_counts.append(sc)
        mean_w_lens.append(np.mean([len(w) for w in words]) if wc > 0 else 0)
        unique_words.append(len(set([w.lower() for w in words])))

    ling_features = np.column_stack([
        word_counts, sent_counts, mean_w_lens, unique_words,
        np.array(word_counts) / np.array(sent_counts)
    ])

    print("Extracting embeddings...")
    embeddings = emb_model.encode(df['transcript'].fillna("").tolist(), show_progress_bar=True)

    return ling_features, embeddings

X_train_ling, X_train_emb = extract_features(df_train)
X_test_ling, X_test_emb = extract_features(df_test)
X_train_comb = np.hstack([X_train_ling, X_train_emb])
X_test_comb = np.hstack([X_test_ling, X_test_emb])
y_train = df_train['label'].values

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

Extracting embeddings...


Batches:   0%|          | 0/25 [00:00<?, ?it/s]

Linguistic:   0%|          | 0/216 [00:00<?, ?it/s]

Extracting embeddings...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

## 5. Model Validation & Cross-Validation

In [ ]:
def evaluate_cv(X, y, model, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    val_rmses, val_pearsons = [], []
    oof_preds = np.zeros(len(y))

    for train_idx, val_idx in kf.split(X):
        X_tr, X_va = X[train_idx], X[val_idx]
        y_tr, y_va = y[train_idx], y[val_idx]

        model.fit(X_tr, y_tr)
        preds = np.clip(model.predict(X_va), 0, 5)
        oof_preds[val_idx] = preds

        rmse = root_mean_squared_error(y_va, preds)
        pearson = 0.0 if np.std(preds) == 0 else pearsonr(y_va, preds)[0]
        val_rmses.append(rmse)
        val_pearsons.append(pearson)

    overall_rmse = root_mean_squared_error(y, oof_preds)
    return np.mean(val_rmses), overall_rmse

experiments = []

# Baseline
base_model = DummyRegressor(strategy="mean")
mean_rmse, oof_rmse = evaluate_cv(X_train_ling, y_train, base_model)
experiments.append(("Baseline (Mean)", mean_rmse, oof_rmse))

# Ridge (Combined)
ridge_model = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=1.0))])
mean_rmse, oof_rmse = evaluate_cv(X_train_comb, y_train, ridge_model)
experiments.append(("Ridge Regression", mean_rmse, oof_rmse))

# SVR (Embeddings)
svr_model = Pipeline([('scaler', StandardScaler()), ('svr', SVR(C=1.0, kernel='rbf'))])
mean_rmse, oof_rmse = evaluate_cv(X_train_emb, y_train, svr_model)
experiments.append(("Support Vector Regression (SVR)", mean_rmse, oof_rmse))

# CatBoost (Linguistic)
cat_model = CatBoostRegressor(verbose=False, random_state=42)
mean_rmse, oof_rmse = evaluate_cv(X_train_ling, y_train, cat_model)
experiments.append(("CatBoost", mean_rmse, oof_rmse))

# XGBoost (Combined)
xgb_model = XGBRegressor(random_state=42)
mean_rmse, oof_rmse = evaluate_cv(X_train_comb, y_train, xgb_model)
experiments.append(("XGBoost", mean_rmse, oof_rmse))

res_df = pd.DataFrame(experiments, columns=["Model", "Mean CV RMSE", "OOF RMSE"]).sort_values("Mean CV RMSE")
display(res_df)

,Model,Mean CV RMSE,OOF RMSE
2,Support Vector Regression (SVR),0.874140,0.876163
3,CatBoost,0.926627,0.926784
4,XGBoost,0.934299,0.935589
1,Ridge Regression,1.180866,1.183731
0,Baseline (Mean),1.239723,1.241407


## TRAINING RMSE � MANDATORY COMPETITION REQUIREMENT

In [ ]:
# Train final model on full dataset
final_model = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=1.0))])
final_model.fit(X_train_comb, y_train)

# Calculate Training RMSE
train_preds = np.clip(final_model.predict(X_train_comb), 0, 5)
train_rmse = root_mean_squared_error(y_train, train_preds)

print("==============================================")
print("TRAINING RMSE � MANDATORY COMPETITION REQUIREMENT")
print(f"Final Model: Ridge Regression (Linguistic + Embeddings)")
print(f"Training RMSE: {train_rmse:.4f}")
print("==============================================")

TRAINING RMSE � MANDATORY COMPETITION REQUIREMENT
Final Model: Ridge Regression (Linguistic + Embeddings)
Training RMSE: 0.5141


## 6. Final Test Prediction

In [ ]:
test_preds = np.clip(final_model.predict(X_test_comb), 0, 5)

df_sub = df_test[['filename']].copy()
df_sub['label'] = test_preds

sub_path = "outputs/FINAL_SUBMISSION.csv"
df_sub.to_csv(sub_path, index=False)
print(f"Submission saved to {sub_path}")
display(df_sub.head())

Submission saved to outputs/FINAL_SUBMISSION.csv


,filename,label
0,audio_128.wav,2.934997
1,audio_156.wav,0.000000
2,audio_19.wav,2.549378
3,audio_108.wav,1.673609
4,audio_206.wav,1.836431
